# Figure 2: beta and beta-binomial distributions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, betabinom, beta as beta_dist
from figstyle import PALETTE, LW, REF_LW, FIGW_SINGLE, FIGW_DOUBLE, save

In [ ]:
# Figure 2: beta threshold densities and beta-binomial degree PMFs at <k> = 5.
MEAN = 5.0
fig, axes = plt.subplots(1, 2, figsize=(FIGW_DOUBLE, 2.9))

# Threshold densities, heterogeneous -> homogeneous
x = np.linspace(0.001, 0.999, 500)
thr_specs = [
    (1, 20,  r'$\mathrm{Beta}(1,20)$'),
    (2, 20,  r'$\mathrm{Beta}(2,20)$'),
    (5, 20,  r'$\mathrm{Beta}(5,20)$'),
    (20, 20, r'$\mathrm{Beta}(20,20)$'),
]
for (a, b, lab), col in zip(thr_specs, PALETTE):
    axes[0].plot(x, beta_dist.pdf(x, a, b), color=col, lw=LW, label=lab)
axes[0].set_xlabel(r'$\phi$')
axes[0].set_ylabel(r'$f(\phi)$')
axes[0].set_xlim(0, 0.8)
axes[0].legend(frameon=False)

# Degree PMFs at <k> = 5, starting at k = 1 (isolates at k = 0 excluded)
kk = np.arange(1, 31)
deg_specs = [
    (0.25, 5,      r'$\alpha_d=0.25$'),
    (1.0,  20,     r'$\alpha_d=1$'),
    (5.0,  95,     r'$\alpha_d=5$'),
    (1000, 100000, r'$\alpha_d=10^3$'),
]
for (ad, bd, lab), col in zip(deg_specs, PALETTE):
    n = max(int(round(MEAN * (ad + bd) / ad)), 2)
    axes[1].plot(kk, betabinom.pmf(kk, n, ad, bd), 'o-',
                 color=col, lw=LW, ms=3, label=lab)
axes[1].set_xlabel(r'$k$')
axes[1].set_ylabel(r'$p_k$')
axes[1].set_xlim(0, 30)
axes[1].legend(frameon=False)

fig.tight_layout()
save(fig, "example_distributions")
plt.show()

In [ ]:
# checks: beta PDFs non-negative, BB PMFs normalize, means match the label
assert all((beta_dist.pdf(x, a, b) >= 0).all() for a, b, _ in thr_specs)
print("beta PDFs non-negative: ok")
var = []
for ad, bd, _ in deg_specs:
    n = max(int(round(MEAN * (ad + bd) / ad)), 2)
    kfull = np.arange(n + 1)
    pk = betabinom.pmf(kfull, n, ad, bd)
    assert abs(pk.sum() - 1) < 1e-9
    mu = (kfull * pk).sum()
    assert abs(mu - MEAN) < 1e-6, f"BB mean {mu} != labeled {MEAN}"
    var.append(((kfull - mu) ** 2 * pk).sum())
    print(f"alpha_d={ad:g}: sum={pk.sum():.12f}, mean={mu:.8f} (label {MEAN:g}), var={var[-1]:.4f}")
assert var[0] > var[1] > var[2] > var[3], "skew progression broken"
print("figure 2 checks passed")